In [1]:
import pandas as pd

# Load all required CSV files
genre_user_info = pd.read_csv("genre_user_info_merged_selected.csv")
processed_data = pd.read_csv("processed_data.csv")
user_info_existing = pd.read_csv("user_info_existing.csv")
merged_user_data = pd.read_csv("merged_user_data.csv")

# Define genre columns
genre_columns = [
    'unknown', 'Action', 'Adventure', 'Animation', "Children's", 'Comedy', 'Crime',
    'Documentary', 'Drama', 'Fantasy', 'Film-Noir', 'Horror', 'Musical', 'Mystery',
    'Romance', 'Sci-Fi', 'Thriller', 'War', 'Western'
]

# Step 1: Genre-wise rating counts per user
genre_counts = processed_data.groupby('user_id')[genre_columns].sum().reset_index()

# Step 2: Total ratings per user
total_rated = processed_data.groupby('user_id')['movie_id'].count().reset_index(name='total_rated')

# Step 3: Combine genre counts and total ratings
ratings_df = pd.merge(total_rated, genre_counts, on='user_id')

# Step 4: Filter users from genre_user_info and merge with ratings info
final_df = pd.merge(genre_user_info[['user_id']], ratings_df, on='user_id', how='left')

# Step 5: Add user demographic data
final_df = pd.merge(final_df, user_info_existing, on='user_id', how='left')

# Step 6: Add flag for user existence in merged_user_data
final_df['exists_in_merged_user_data'] = final_df['user_id'].isin(merged_user_data['user_id'])

# Step 7: Add files_present_in column (fill 0 for non-existent users)
if 'files_present_in' in merged_user_data.columns:
    files_info = merged_user_data[['user_id', 'files_present_in']]
    final_df = pd.merge(final_df, files_info, on='user_id', how='left')
    final_df['files_present_in'] = final_df['files_present_in'].fillna(0)
else:
    final_df['files_present_in'] = 0

# Step 8: Reorder columns
column_order = ['user_id', 'gender', 'occupation', 'zip_code', 'age', 'total_rated'] + genre_columns[1:] + ['exists_in_merged_user_data', 'files_present_in']
final_df = final_df[column_order]

# Step 9: Save final output
final_df.to_csv("user_genre_rating_summary.csv", index=False)
print("✅ File saved as user_genre_rating_summary.csv")


✅ File saved as user_genre_rating_summary.csv
